# 02 — Static Gesture Recognition — Training

> **GPU Required** — Run on CUDA-capable hardware.

**Pipeline:**
1. Load static landmark arrays
2. Define MobileNetV3-style MLP classifier
3. Train with: label smoothing · cosine annealing · EMA · early stopping · mixed precision
4. Evaluate on validation & test sets
5. Export to ONNX


In [ ]:
import os, json, time
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import onnx

ROOT            = Path("..")
DATA_LANDMARKS  = ROOT / "data" / "landmarks" / "static"
MODELS_DIR      = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────
CFG = dict(
    input_dim       = 126,      # 2 hands × 21 × 3
    hidden_dims     = [512, 256, 128],
    dropout         = 0.3,
    label_smoothing = 0.1,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    batch_size      = 256,
    epochs          = 100,
    patience        = 12,       # early stopping
    ema_decay       = 0.999,
    warmup_epochs   = 5,
)
print(CFG)

In [ ]:
# ── Load data ─────────────────────────────────────────────────
def load_split(split):
    p = DATA_LANDMARKS / split
    X = torch.tensor(np.load(p / "X.npy"), dtype=torch.float32)
    y = torch.tensor(np.load(p / "y.npy"), dtype=torch.long)
    return X, y

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")
X_test,  y_test  = load_split("test")

NUM_CLASSES = int(y_train.max().item()) + 1
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Classes: {NUM_CLASSES}")

train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=CFG["batch_size"], shuffle=True,  drop_last=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),
                          batch_size=CFG["batch_size"], shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),
                          batch_size=CFG["batch_size"], shuffle=False)

In [ ]:
# ── Model definition ──────────────────────────────────────────
class StaticClassifier(nn.Module):
    """MLP classifier for single-frame hand landmarks."""

    def __init__(self, input_dim: int, hidden_dims: list, num_classes: int, dropout: float):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = StaticClassifier(
    CFG["input_dim"], CFG["hidden_dims"], NUM_CLASSES, CFG["dropout"]
).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

In [ ]:
# ── Exponential Moving Average ────────────────────────────────
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)

    def apply_shadow(self, model):
        model.load_state_dict(self.shadow)

ema = EMA(model, decay=CFG["ema_decay"])

In [ ]:
# ── Training ──────────────────────────────────────────────────
criterion  = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
optimizer  = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
scaler     = GradScaler()
writer     = SummaryWriter(log_dir=str(ROOT / "logs" / "static"))

best_val_acc = 0.0
best_ckpt    = None
patience_ctr = 0
history      = {"train_loss": [], "val_acc": []}

for epoch in range(1, CFG["epochs"] + 1):
    # ── Train ────────────────────────────────────
    model.train()
    total_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            loss = criterion(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(train_loader)

    # ── Validate ─────────────────────────────────
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(DEVICE)
            out = model(X_b).argmax(dim=1).cpu()
            preds.append(out); truths.append(y_b)
    val_acc = accuracy_score(torch.cat(truths), torch.cat(preds))

    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_ckpt    = deepcopy(model.state_dict())
        patience_ctr = 0
        print(f"  ★ Epoch {epoch:3d} | loss {avg_loss:.4f} | val_acc {val_acc:.4f}  ← best")
    else:
        patience_ctr += 1
        if epoch % 10 == 0:
            print(f"    Epoch {epoch:3d} | loss {avg_loss:.4f} | val_acc {val_acc:.4f}")

    if patience_ctr >= CFG["patience"]:
        print(f"\nEarly stopping at epoch {epoch}")
        break

writer.close()
print(f"\nBest val accuracy: {best_val_acc:.4f}")

In [ ]:
# ── Test evaluation ───────────────────────────────────────────
model.load_state_dict(best_ckpt)
ema.apply_shadow(model)
model.eval()

preds, truths = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        preds.append(model(X_b.to(DEVICE)).argmax(1).cpu())
        truths.append(y_b)

preds  = torch.cat(preds).numpy()
truths = torch.cat(truths).numpy()
print(classification_report(truths, preds, zero_division=0))

In [ ]:
# ── Training curves ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], color="steelblue")
ax1.set_title("Training Loss"); ax1.set_xlabel("Epoch")
ax2.plot(history["val_acc"],   color="darkorange")
ax2.set_title("Validation Accuracy"); ax2.set_xlabel("Epoch")
plt.tight_layout(); plt.show()

In [ ]:
# ── ONNX Export ───────────────────────────────────────────────
model.eval().to("cpu")
dummy = torch.zeros(1, CFG["input_dim"])
onnx_path = str(MODELS_DIR / "arabic_sign_model_medium.onnx")

torch.onnx.export(
    model, dummy, onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["landmarks"],
    output_names=["logits"],
    dynamic_axes={"landmarks": {0: "batch"}, "logits": {0: "batch"}},
)

# Verify
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"ONNX model saved → {onnx_path}")